# NB07c — Base-paper beat attempt (facebook-44001, native 5-class)

**Goal:** on the base paper's own protocol (facebook-44001, within-source dedup, native 5 classes,
70/15/15 stratified) push a single-encoder Model-B–style system past the base paper's **Macro-F1 0.8923**.

**What's different from 07b (by design):**
- **Recipe = attention-pooling + LLRD + plain cross-entropy + R-Drop + MSD + EMA(val-guarded), max_length=192.**
  No balanced sampler, no focal, no FGM, no AWP — this is exactly the spec agreed on.
- **Backbones run in sequence:** (1) `csebuetnlp/banglishbert`, (2) `xlm-roberta-large`, each with **3 seeds**.
- **Two ensembles reported:** per-backbone seed-ensemble, then a **cross-backbone ensemble** (prob-space,
  Nelder–Mead weights fit on val, applied to test).
- **The split is built once and saved to disk.** Every backbone and every seed loads the *same* saved
  train/val/test CSVs, so there is a single fixed test set and zero train/test contamination.

**Honest expectation:** ~0.88–0.90 Macro-F1. Beating 0.8923 overall is ~50–60% likely; the near-certain
win is the `Threat` class (already 0.8337 vs 0.7579). Do **not** switch on the balanced sampler or
oversample-before-split to inflate this — that was the leakage path.

Run **DEBUG=True first** (2 epochs, tiny subset) to verify end-to-end, then set DEBUG=False.


In [1]:
# === DEPENDENCIES (RunPod PyTorch image already ships torch+CUDA — do NOT reinstall torch) ===
# sentencepiece is REQUIRED for the XLM-R tokenizer; accelerate is used by transformers internals.
import sys, subprocess
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","--quiet",*a],check=True)
pip("transformers==4.44.2","accelerate==0.33.0")
pip("sentencepiece==0.2.0","scikit-learn==1.5.1","scipy==1.13.1","pandas==2.2.2","numpy==1.26.4")
# If your RunPod image has torch<2.1 (no torch.amp / weights_only), tell me and I'll patch those calls.
print("deps ok")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.4 requires pyarrow>=21.0.0, but you have pyarrow 20.0.0 which is incompatible.
pandas-profiling 3.2.0 requires joblib~=1.1.0, but you have joblib 1.5.3 which is incompatible.
tensorflow 2.18.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
ydata-profiling 4.18.1 requires visions[type_image_path]<0.8.2,>=0.7.5, but you have visions 0.7.4 which is incompatible.
mlxtend 0.24.0 requires matplotlib>=3.10.8, but you have matplotlib 3.10.0 which is incompatible.
mlxtend 0.24.0 requires numpy>=2.3.5, but you have numpy 1.26.4 which is incompatible.
mlxtend 0.24.0 requires pandas>=2.3.3, but you have pandas 2.2.3 which is incompatible.
mlxtend 0.24.0 requires scikit-learn>=1.8.0, but you have scikit-learn 1.7.2 whic

CalledProcessError: Command '['/Users/sefayet/.pyenv/versions/3.11.6/bin/python', '-m', 'pip', 'install', '--quiet', 'sentencepiece==0.2.0', 'scikit-learn==1.5.1', 'scipy==1.13.1', 'pandas==2.2.2', 'numpy==1.26.4']' returned non-zero exit status 2.

In [ ]:
# === ENVIRONMENT ===
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"torch: {torch.__version__}")

In [ ]:
import os, re, json, math, time, random, warnings, unicodedata
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, roc_auc_score
from scipy.optimize import minimize
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# === CONFIG (edit PATHS if your layout differs) ===============================
DEBUG = False                 # <-- run True first (fast smoke test), then set False for the full run
DEBUG_SAMPLES = 500
DEBUG_EPOCHS  = 2

# Paths — match the rest of your repo (notebook sits in notebooks/, data in ../data, outputs in ../outputs)
RAW        = "../data/merged/benchmark_raw.csv"     # same source file 07b/NB09 use
SPLIT_OUT  = "../data/splits_basepaper"             # the SAVED base-paper split lives here (built once)
OUT        = "../outputs/altmethod_07c"
os.makedirs(SPLIT_OUT, exist_ok=True); os.makedirs(OUT, exist_ok=True)
SPLIT_SEED = 42
SEEDS      = [42, 123, 456]                          # 3 seeds per backbone

# Recipe toggles — exactly the agreed spec. Sampler / focal / FGM / AWP are intentionally OFF.
CFG = {
    "max_length": 192, "eval_batch_size": 128, "num_workers": 2,
    "epochs": 8, "patience": 3, "weight_decay": 0.01, "warmup_ratio": 0.10,
    "max_grad_norm": 1.0, "label_smoothing": 0.0,      # 0.0 = pure cross-entropy (spec). 0.03 sometimes helps.
    "dropout": 0.25, "head_hidden_dim": 384, "n_msd": 4,
    "llrd_decay": 0.90,                                 # depth-wise LR decay (orthogonal to the time schedule)
    "use_rdrop": True, "rdrop_alpha": 0.5,
    "use_ema": True, "ema_decay": 0.999,
    # ---- adversarial training (OFF by default; turn ON one at a time to A/B) ----
    "use_fgm": True, "fgm_epsilon": 1.0,
    "use_awp": True, "awp_lr": 1.0, "awp_eps": 0.01, "awp_start_epoch": 1,
    "use_fp16": torch.cuda.is_available(),
    # ---- safety valves, OFF by design; flip ONE of these on only if Threat/Religious recall sags ----
    "use_class_weights": False, "class_weight_beta": 0.999,
    "logit_adjust_tau": 0.0,                            # >0 shifts logits by tau*log(prior) at INFERENCE only
}

# Per-backbone settings. XLM-R-large is 560M → smaller physical batch + grad-accum + grad-checkpointing,
# and a lower encoder LR (large models fine-tune more stably at 1e-5). Effective batch ~32 for both.
BACKBONES = [
    {"tag":"banglishbert", "hf_id":"csebuetnlp/banglishbert", "batch_size":32, "grad_accum":1,
     "encoder_lr":2e-5, "head_lr":8e-5, "grad_ckpt":False},
    {"tag":"xlmr_large",   "hf_id":"xlm-roberta-large",       "batch_size":8,  "grad_accum":4,
     "encoder_lr":1e-5, "head_lr":1e-4, "grad_ckpt":True},
]

BASE_PAPER = {"macro_f1":0.8923,
              "per_class":{"Not Bully":0.9151,"Sexual":0.8845,"Troll":0.8446,"Religious":0.9374,"Threat":0.7579}}

if DEBUG:
    CFG["epochs"] = DEBUG_EPOCHS; SEEDS = [42]
    print("⚠ DEBUG MODE: 1 seed, 2 epochs, tiny subset")
print(f"seeds={SEEDS} | max_len={CFG['max_length']} | llrd={CFG['llrd_decay']} "
      f"| rdrop={CFG['use_rdrop']} msd={CFG['n_msd']} ema={CFG['use_ema']} | sampler/focal/fgm/awp = OFF")

In [ ]:
# === MACHINERY (attention pooling + LLRD + CE + R-Drop + MSD + EMA) ==========
class DS(Dataset):
    def __init__(self, df, tok, maxlen, enc, text_col, label_col):
        self.texts = df.reset_index(drop=True)[text_col].fillna("").astype(str).tolist()
        self.labels = [int(enc.get(v,-1)) for v in df[label_col]]
        self.tok, self.maxlen = tok, maxlen
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        e = self.tok(self.texts[i], max_length=self.maxlen, truncation=True, padding=False)
        it = {k: torch.tensor(v, dtype=torch.long) for k,v in e.items()}
        it["label"] = torch.tensor(self.labels[i], dtype=torch.long); return it

class Collator:
    def __init__(self, tok): self.tok = tok
    def __call__(self, fs):
        labels = torch.stack([f["label"] for f in fs])
        feats = [{k:v for k,v in f.items() if k!="label"} for f in fs]
        b = self.tok.pad(feats, padding=True, return_tensors="pt"); b["label"] = labels; return b

class AttnPool(nn.Module):
    # learned masked attention over token states: replaces 07b's fixed 0.5*CLS + 0.5*mean pooling
    def __init__(self, h, ad=256):
        super().__init__(); ad = min(ad, h)
        self.W = nn.Linear(h, ad); self.v = nn.Linear(ad, 1, bias=False)
    def forward(self, hs, mask):
        s = self.v(torch.tanh(self.W(hs))).squeeze(-1)         # (B,L)
        s = s.masked_fill(mask == 0, float("-inf"))
        a = torch.softmax(s, -1).unsqueeze(-1)                 # (B,L,1)
        return (a * hs).sum(1)                                 # (B,H)

class MSDHead(nn.Module):
    def __init__(self, h, n, dropout=0.25, inner=384, n_msd=4):
        super().__init__(); inner = min(inner, h)
        self.pre = nn.Sequential(nn.Linear(h, inner), nn.GELU(), nn.LayerNorm(inner))
        self.drops = nn.ModuleList([nn.Dropout(dropout) for _ in range(max(1, n_msd))])
        self.out = nn.Linear(inner, n)
    def forward(self, x):
        h = self.pre(x)
        if self.training and len(self.drops) > 1:
            return torch.stack([self.out(d(h)) for d in self.drops], 0).mean(0)
        return self.out(self.drops[0](h))

class Encoder(nn.Module):
    def __init__(self, name, n, dropout=0.25, inner=384, n_msd=4, grad_ckpt=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        if grad_ckpt:
            try:
                self.encoder.config.use_cache = False
                self.encoder.gradient_checkpointing_enable()
            except Exception as e:
                print("  (grad-ckpt not enabled:", e, ")")
        h = self.encoder.config.hidden_size
        # RoBERTa / XLM-R have no token_type_ids; ELECTRA-based BanglishBERT does
        self._tti = self.encoder.config.model_type.lower() not in ("roberta", "xlm-roberta")
        self.attn_pool = AttnPool(h)
        self.head = MSDHead(h, n, dropout, inner, n_msd)
    def forward(self, ids, mask, tti=None):
        kw = {"input_ids": ids, "attention_mask": mask}
        if self._tti and tti is not None: kw["token_type_ids"] = tti
        o = self.encoder(**kw)
        return self.head(self.attn_pool(o.last_hidden_state, mask))

def llrd_params(model, encoder_lr, head_lr, wd, decay):
    # Depth-wise (layer-wise) LR decay. This is NOT the time-wise decay your ablation rejected —
    # top layer keeps encoder_lr, each lower layer is scaled by decay, embeddings lowest.
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias", "layer_norm.weight", "layer_norm.bias")
    L = model.encoder.config.num_hidden_layers
    layer_re = re.compile(r"\.layer\.(\d+)\.")
    def lr_for(name):
        if name.startswith("head") or name.startswith("attn_pool"): return head_lr
        m = layer_re.search(name)
        if m: return encoder_lr * (decay ** ((L-1) - int(m.group(1))))
        if "embeddings" in name: return encoder_lr * (decay ** L)
        return encoder_lr                                       # pooler / projections / leftovers
    buckets = {}
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        lr = lr_for(n); w = 0.0 if any(x in n for x in no_decay) else wd
        buckets.setdefault((round(lr, 12), w), []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": w} for (lr, w), ps in buckets.items()]

def build_cw(series, enc, beta=0.999, max_w=10.0):
    m = series.map(enc).dropna().astype(int); n = len(enc); c = m.value_counts().sort_index()
    w = np.ones(n, dtype=np.float32)
    for i in range(n):
        k = int(c.get(i, 0))
        if k > 0: w[i] = 1.0 / max((1.0 - beta**k)/max(1.0-beta, 1e-12), 1e-9)
    w = np.clip(w, w.min(), w.min()*max_w); w = w/w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

class EMA:
    def __init__(self, model, decay=0.999):
        self.d = decay
        self.s = {n: p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}; self.b = {}
    def update(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad: self.s[n].mul_(self.d).add_(p.detach(), alpha=1-self.d)
    def apply_shadow(self, model):
        self.b = {n: p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
        for n,p in model.named_parameters():
            if p.requires_grad: p.data.copy_(self.s[n])
    def restore(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad and n in self.b: p.data.copy_(self.b[n])
        self.b = {}

def rdrop_kl(lp, lq):
    pl, ql = F.log_softmax(lp,-1), F.log_softmax(lq,-1); p, q = pl.exp(), ql.exp()
    return 0.5*((p*(pl-ql)).sum(-1) + (q*(ql-pl)).sum(-1)).mean()

@torch.no_grad()
def get_logits(model, loader):
    model.eval(); o = []
    for b in loader:
        b = {k: v.to(device) for k,v in b.items()}
        o.append(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")).cpu())
    return torch.cat(o)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def metrics(yt, yp, pr, nc):
    rec = {"macro_f1": round(float(f1_score(yt,yp,average="macro",zero_division=0)),4),
           "weighted_f1": round(float(f1_score(yt,yp,average="weighted",zero_division=0)),4),
           "accuracy": round(float(accuracy_score(yt,yp)),4),
           "mcc": round(float(matthews_corrcoef(yt,yp)),4)}
    try: rec["macro_auroc"] = round(float(roc_auc_score(yt,pr,multi_class="ovr",average="macro",labels=list(range(nc)))),4)
    except Exception: rec["macro_auroc"] = float("nan")
    return rec
print("machinery ok")

In [ ]:
# === ADVERSARIAL TRAINING: FGM (embedding-space) + AWP (weight-space) ========
class FGM:
    # perturbs the token-embedding matrix along its gradient; cheap, scale-invariant
    def __init__(self, model, eps=1.0, key="word_embeddings"):
        self.m, self.e, self.k, self.b = model, eps, key, {}
    def attack(self):
        for n,p in self.m.named_parameters():
            if p.requires_grad and self.k in n and p.grad is not None:
                self.b[n] = p.data.clone(); nm = torch.norm(p.grad)
                if nm != 0 and not torch.isnan(nm): p.data.add_(self.e*p.grad/nm)
    def restore(self):
        for n,p in self.m.named_parameters():
            if n in self.b: p.data = self.b[n]
        self.b = {}

class AWP:
    # perturbs encoder WEIGHT matrices (not embeddings) along gradient ascent, clipped to
    # adv_eps*||w||; stronger regularizer than FGM. Skips embeddings/LayerNorm/head.
    def __init__(self, model, adv_lr=1.0, adv_eps=0.01, adv_param="weight", key="encoder"):
        self.m, self.adv_lr, self.adv_eps, self.adv_param, self.key = model, adv_lr, adv_eps, adv_param, key
        self.bak, self.bak_eps = {}, {}
    def _hit(self, n, p):
        return (p.requires_grad and p.grad is not None and self.adv_param in n and self.key in n
                and "embeddings" not in n and "LayerNorm" not in n and "layer_norm" not in n)
    def attack(self):
        e = 1e-6
        for n,p in self.m.named_parameters():
            if self._hit(n,p):
                self.bak[n] = p.data.clone()
                gn = torch.norm(p.grad); dn = torch.norm(p.data.detach())
                if gn != 0 and not torch.isnan(gn):
                    self.bak_eps[n] = (self.bak[n]-self.adv_eps*(dn+e), self.bak[n]+self.adv_eps*(dn+e))
                    p.data.add_(self.adv_lr*p.grad/(gn+e)*(dn+e))
                    p.data = torch.min(torch.max(p.data, self.bak_eps[n][0]), self.bak_eps[n][1])
    def restore(self):
        for n,p in self.m.named_parameters():
            if n in self.bak: p.data = self.bak[n]
        self.bak, self.bak_eps = {}, {}
print("FGM + AWP ok")

In [ ]:
# === SANITY CHECK: confirm FGM/AWP actually match parameters (catches silent no-ops) ===
def _check_adv_coverage(hf_id):
    m = Encoder(hf_id, NC, CFG["dropout"], CFG["head_hidden_dim"], CFG["n_msd"]).to(device)
    fgm_hits = sum(1 for n,p in m.named_parameters() if "word_embeddings" in n)
    awp_hits = sum(1 for n,p in m.named_parameters()
                   if "weight" in n and "encoder" in n
                   and "embeddings" not in n and "LayerNorm" not in n and "layer_norm" not in n)
    print(f"  {hf_id}: FGM would touch {fgm_hits} param tensor(s), AWP would touch {awp_hits} param tensor(s)")
    del m; torch.cuda.empty_cache()
    assert fgm_hits > 0 and awp_hits > 0, f"⚠ zero matches for {hf_id} — FGM/AWP would be silent no-ops!"

for bk in BACKBONES: _check_adv_coverage(bk["hf_id"])
print("adv coverage ok")

In [ ]:
# === BUILD + SAVE THE SPLIT ONCE (facebook-44001, native 5-class, 70/15/15) ===
# Built once and written to SPLIT_OUT. Every backbone/seed reloads THESE files, so there is one
# fixed test set and no train/test contamination. Delete the CSVs to force a rebuild.
LABEL = "label5b"; TEXT = "text_clean"
tr_p, va_p, te_p = [f"{SPLIT_OUT}/{s}.csv" for s in ("train","val","test")]

if all(os.path.exists(p) for p in (tr_p, va_p, te_p)):
    trf, vaf, tef = pd.read_csv(tr_p), pd.read_csv(va_p), pd.read_csv(te_p)
    print(f"loaded SAVED split: train={len(trf):,} val={len(vaf):,} test={len(tef):,}")
else:
    df = pd.read_csv(RAW)
    fb = df[df["source"]=="facebook_44001"].reset_index(drop=True).copy()
    print(f"facebook rows: {len(fb):,}")
    URL=re.compile(r"https?://\S+|www\.\S+"); MEN=re.compile(r"@[\w]+"); HASH=re.compile(r"#(\S+)")
    ZW=re.compile(r"[\u200b\u200c\u200d\ufeff]"); WS=re.compile(r"\s+"); ELONG=re.compile(r"(.)\1{2,}")
    EMO=re.compile("["+"\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
                   "\U0001F1E0-\U0001F1FF\U00002700-\U000027BF\U0001F900-\U0001F9FF\U00002600-\U000026FF]+",flags=re.UNICODE)
    def clean(t):
        if not isinstance(t,str): return ""
        t=unicodedata.normalize("NFKC",t); t=ZW.sub("",t); t=URL.sub(" [URL] ",t); t=MEN.sub(" [USER] ",t)
        t=HASH.sub(r" [HASHTAG] \1 ",t); t=EMO.sub(" [EMOJI] ",t); t=ELONG.sub(r"\1\1",t); return WS.sub(" ",t).strip()
    fb[TEXT] = fb["text"].apply(clean); fb = fb[fb[TEXT].str.len()>0]
    n0=len(fb); fb = fb.drop_duplicates(TEXT).reset_index(drop=True)
    print(f"within-facebook dedup: {n0:,} → {len(fb):,}")
    NAME={"not bully":"Not Bully","sexual":"Sexual","troll":"Troll","religious":"Religious","threat":"Threat"}
    fb[LABEL] = fb["label_type"].astype(str).str.strip().str.lower().map(NAME)
    fb = fb[fb[LABEL].notna()].reset_index(drop=True); assert fb[LABEL].nunique()==5
    trf, tmp = train_test_split(fb,  test_size=0.30, random_state=SPLIT_SEED, stratify=fb[LABEL])
    vaf, tef = train_test_split(tmp, test_size=0.50, random_state=SPLIT_SEED, stratify=tmp[LABEL])
    keep=[TEXT, LABEL]
    trf[keep].to_csv(tr_p, index=False); vaf[keep].to_csv(va_p, index=False); tef[keep].to_csv(te_p, index=False)
    print(f"BUILT + SAVED split → {SPLIT_OUT}  train={len(trf):,} val={len(vaf):,} test={len(tef):,}")

classes = sorted(pd.concat([trf,vaf,tef])[LABEL].unique()); enc = {c:i for i,c in enumerate(classes)}
NC = len(classes); print("classes:", enc)
print("test class balance:\n", tef[LABEL].value_counts())

if DEBUG:  # subsample the SAVED split (never re-split) so debug still uses the fixed test set
    trf = pd.concat([g.sample(min(len(g), DEBUG_SAMPLES//NC), random_state=42) for _,g in trf.groupby(LABEL)]).reset_index(drop=True)
    vaf = vaf.sample(min(len(vaf),300), random_state=42); tef = tef.sample(min(len(tef),300), random_state=42)
    print(f"DEBUG subset → train={len(trf):,} val={len(vaf):,} test={len(tef):,}")

In [ ]:
# === TRAIN ONE (CE + R-Drop + MSD + EMA + LLRD [+ optional FGM/AWP]); saves logits =====
def train_one(seed, bk, train_df, val_df, test_df, enc, nc, save, text_col=TEXT, label_col=LABEL):
    set_seed(seed); os.makedirs(save, exist_ok=True); cfg = CFG
    hf_id=bk["hf_id"]; bs=bk["batch_size"]; accum=bk["grad_accum"]
    elr=bk["encoder_lr"]; hlr=bk["head_lr"]; gckpt=bk.get("grad_ckpt", False)
    ckpts = [f"{save}/best.pt", f"{save}/val_logits.pt", f"{save}/test_logits.pt"]
    if all(os.path.exists(p) for p in ckpts):
        print(f"    seed{seed}: ⏭ checkpoint found, skipping"); return json.load(open(f"{save}/run.json"))["best_val"]
    tok = AutoTokenizer.from_pretrained(hf_id); coll = Collator(tok)
    lkw = dict(collate_fn=coll, num_workers=cfg["num_workers"], pin_memory=torch.cuda.is_available())
    tl = DataLoader(DS(train_df,tok,cfg["max_length"],enc,text_col,label_col),
                    batch_size=bs, shuffle=True, drop_last=True, **lkw)
    vl = DataLoader(DS(val_df,tok,cfg["max_length"],enc,text_col,label_col),
                    batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)
    model = Encoder(hf_id, nc, cfg["dropout"], cfg["head_hidden_dim"], cfg["n_msd"], grad_ckpt=gckpt).to(device)
    cw = build_cw(train_df[label_col], enc, cfg["class_weight_beta"]) if cfg["use_class_weights"] else None
    def crit(lg, t): return F.cross_entropy(lg, t, weight=cw, label_smoothing=cfg["label_smoothing"])
    opt = torch.optim.AdamW(llrd_params(model, elr, hlr, cfg["weight_decay"], cfg["llrd_decay"]))
    steps = math.ceil(len(tl)/accum)*cfg["epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(steps*cfg["warmup_ratio"]), steps)
    scaler = torch.amp.GradScaler("cuda") if (cfg["use_fp16"] and device.type=="cuda") else None
    ema = EMA(model, cfg["ema_decay"]) if cfg["use_ema"] else None
    fgm = FGM(model, cfg["fgm_epsilon"]) if cfg.get("use_fgm") else None
    awp = AWP(model, cfg["awp_lr"], cfg["awp_eps"]) if cfg.get("use_awp") else None
    @torch.no_grad()
    def vmacro():
        model.eval(); P,Y=[],[]
        for b in vl:
            b={k:v.to(device) for k,v in b.items()}
            P.extend(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")).argmax(-1).cpu().numpy())
            Y.extend(b["label"].cpu().numpy())
        Y=np.array(Y); m=Y>=0; return f1_score(Y[m], np.array(P)[m], average="macro", zero_division=0)
    best, noimp, t0 = -1.0, 0, time.time()
    for ep in range(cfg["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True)
        for step, b in enumerate(tl, 1):
            b={k:v.to(device) for k,v in b.items()}; y=b["label"]
            with torch.autocast(device_type=device.type, enabled=scaler is not None):
                l1 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                if cfg["use_rdrop"]:
                    l2 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                    loss = 0.5*(crit(l1,y)+crit(l2,y)) + cfg["rdrop_alpha"]*rdrop_kl(l1,l2)
                else:
                    loss = crit(l1, y)
                loss = loss/accum
            (scaler.scale(loss) if scaler else loss).backward()
            # --- FGM: embedding-space adversarial pass ---
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=device.type, enabled=scaler is not None):
                    la = crit(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")), y)/accum
                (scaler.scale(la) if scaler else la).backward(); fgm.restore()
            # --- AWP: weight-space adversarial pass (gated by warmup epoch) ---
            if awp is not None and ep >= cfg["awp_start_epoch"]:
                awp.attack()
                with torch.autocast(device_type=device.type, enabled=scaler is not None):
                    la2 = crit(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")), y)/accum
                (scaler.scale(la2) if scaler else la2).backward(); awp.restore()
            if step % accum == 0 or step == len(tl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                (scaler.step(opt), scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(model)
        vm_raw = vmacro(); vm, use_ema = vm_raw, False
        if ema is not None and ep >= 2:
            ema.apply_shadow(model); vm_ema = vmacro()
            if vm_ema >= vm_raw: vm, use_ema = vm_ema, True
            ema.restore(model)
        tag = "＋ema" if use_ema else "    "
        print(f"    seed{seed} ep{ep+1}: val_macroF1={vm:.4f} {tag}")
        if vm > best:
            best, noimp = vm, 0
            if use_ema: ema.apply_shadow(model)
            torch.save(model.state_dict(), f"{save}/best.pt")
            if use_ema: ema.restore(model)
        else:
            noimp += 1
        if noimp >= cfg["patience"]: print(f"    early stop @ ep{ep+1}"); break
    model.load_state_dict(torch.load(f"{save}/best.pt", map_location=device, weights_only=True))
    tel = DataLoader(DS(test_df,tok,cfg["max_length"],enc,text_col,label_col),
                     batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)
    torch.save(get_logits(model, vl),  f"{save}/val_logits.pt")
    torch.save(get_logits(model, tel), f"{save}/test_logits.pt")
    json.dump({"best_val": float(best), "seed": seed, "backbone": bk["tag"]}, open(f"{save}/run.json","w"))
    print(f"    seed{seed}: best_val_macroF1={best:.4f}  [{(time.time()-t0)/60:.1f} min]")
    del model; torch.cuda.empty_cache()
    return float(best)
print("train_one ok")

In [ ]:
# === ENSEMBLE HELPERS (probability-space; safe across different backbones) ====
def apply_tau(lg, tau, prior):
    if tau and tau != 0.0:
        return lg + tau*torch.log(torch.tensor(prior, dtype=lg.dtype)).unsqueeze(0)
    return lg
def probs_of(lg, prior=None):
    return torch.softmax(apply_tau(lg, CFG["logit_adjust_tau"], prior) if prior is not None else lg, -1).numpy()
def ens_probs(logit_dict, names, w, prior=None):
    P=None
    for i,n in enumerate(names):
        p = probs_of(logit_dict[n], prior)
        P = w[i]*p if P is None else P + w[i]*p
    return P/(np.sum(w)+1e-12)
def opt_weights(val_dict, names, y_val, prior=None):
    k=len(names)
    if k==1: return np.ones(1)
    best, bw = 1.0, np.ones(k)/k
    for _ in range(40):
        r = minimize(lambda rw: -f1_score(y_val, ens_probs(val_dict,names,np.abs(rw)+1e-9,prior).argmax(-1),
                                          average="macro", zero_division=0),
                     np.random.dirichlet(np.ones(k)), method="Nelder-Mead",
                     options={"maxiter":2000,"xatol":1e-5})
        if r.fun < best: best, bw = r.fun, np.abs(r.x)+1e-9
    return bw/bw.sum()

y_val  = vaf[LABEL].map(enc).values
y_test = tef[LABEL].map(enc).values
prior  = np.bincount(trf[LABEL].map(enc).values, minlength=NC).astype(float); prior = prior/prior.sum()
inv = {i:c for c,i in enc.items()}

def report(P, tag):
    yp = P.argmax(-1); m = metrics(y_test, yp, P, NC)
    pcf = f1_score(y_test, yp, average=None, labels=list(range(NC)), zero_division=0)
    m["per_class_f1"] = {inv[i]: round(float(pcf[i]),4) for i in range(NC)}
    d = m["macro_f1"] - BASE_PAPER["macro_f1"]
    print(f"\n[{tag}] Macro-F1 {m['macro_f1']:.4f} | base {BASE_PAPER['macro_f1']:.4f} | Δ {d:+.4f}"
          f" | Wtd {m['weighted_f1']} Acc {m['accuracy']} MCC {m['mcc']} AUROC {m['macro_auroc']}")
    for c in classes:
        print(f"    {c:12s} {m['per_class_f1'][c]:.4f}  vs base {BASE_PAPER['per_class'].get(c,float('nan')):.4f}")
    return m
print("ensemble helpers ok")

In [ ]:
# === BACKBONE 1: BanglishBERT (3 seeds) ======================================
ALL_VAL, ALL_TEST = {}, {}    # collected across BOTH backbones for the cross-backbone ensemble

def run_backbone(bk):
    root = f"{OUT}/{bk['tag']}"; os.makedirs(root, exist_ok=True)
    vdict, tdict = {}, {}
    print(f"\n{'='*70}\nBACKBONE: {bk['tag']}  ({bk['hf_id']})\n{'='*70}")
    for sd in SEEDS:
        print(f"  seed {sd}:")
        train_one(sd, bk, trf, vaf, tef, enc, NC, f"{root}/seed{sd}")
        name = f"{bk['tag']}_s{sd}"
        vdict[name] = torch.load(f"{root}/seed{sd}/val_logits.pt",  map_location="cpu", weights_only=False).float()
        tdict[name] = torch.load(f"{root}/seed{sd}/test_logits.pt", map_location="cpu", weights_only=False).float()
    names = list(vdict.keys())
    W = opt_weights(vdict, names, y_val, prior)
    Pt = ens_probs(tdict, names, W, prior)
    m = report(Pt, f"{bk['tag']} seed-ensemble")
    m["weights"] = {n: float(w) for n,w in zip(names, W)}
    json.dump(m, open(f"{root}/seed_ensemble_metrics.json","w"), indent=2)
    ALL_VAL.update(vdict); ALL_TEST.update(tdict)      # feed the cross-backbone ensemble
    return m

m_bb = run_backbone(BACKBONES[0])

In [ ]:
# === BACKBONE 2: XLM-R-large (3 seeds) =======================================
# Heaviest cell. If you hit CUDA OOM, lower BACKBONES[1]["batch_size"] to 4 and raise "grad_accum" to 8.
#m_xlmr = run_backbone(BACKBONES[1])

In [ ]:
# === CROSS-BACKBONE ENSEMBLE + BASE-PAPER VERDICT ============================
'''
names = list(ALL_VAL.keys())
print("runs in ensemble:", names)
W = opt_weights(ALL_VAL, names, y_val, prior)
P_final = ens_probs(ALL_TEST, names, W, prior)
m_final = report(P_final, "CROSS-BACKBONE ENSEMBLE (final)")
m_final["weights"] = {n: float(w) for n,w in zip(names, W)}
m_final["n_test"] = int(len(y_test)); m_final["seeds"] = SEEDS
m_final["backbones"] = [b["tag"] for b in BACKBONES]

summary = {
    "final_cross_backbone": m_final,
    "banglishbert_seed_ensemble": m_bb,
    "xlmr_large_seed_ensemble": m_xlmr,
    "base_paper": BASE_PAPER,
    "beats_base_overall": bool(m_final["macro_f1"] > BASE_PAPER["macro_f1"]),
}
json.dump(summary, open(f"{OUT}/basepaper_comparison_07c.json","w"), indent=2)

print("\n" + "="*70)
print(f"VERDICT: final Macro-F1 {m_final['macro_f1']:.4f} vs base {BASE_PAPER['macro_f1']:.4f}"
      f"  →  {'BEATS' if summary['beats_base_overall'] else 'below'} overall")
thr = m_final["per_class_f1"]["Threat"]
print(f"Threat class: {thr:.4f} vs base {BASE_PAPER['per_class']['Threat']:.4f}  (Δ {thr-BASE_PAPER['per_class']['Threat']:+.4f})")
print(f"✅ saved {OUT}/basepaper_comparison_07c.json")
'''

---
**Read-out guide**

- `Δ` next to Macro-F1 is vs the base paper (0.8923). Positive = you beat it overall.
- Even if overall Δ is slightly negative, the **Threat** line is the defensible headline (you already lead there).
- The saved split in `../data/splits_basepaper/` is now the single source of truth — reuse it for any
  follow-up (ablation, error analysis) so every number is on the identical test set.
- Cost control: every seed writes `best.pt` + `val_logits.pt` + `test_logits.pt`, so re-running the
  ensemble cell (or resuming after a crash) does **not** retrain finished seeds.
- If Threat/Religious recall looks weak under pure CE, flip **one** valve in CONFIG: either
  `use_class_weights=True` **or** `logit_adjust_tau≈1.0` — not both.
